# 🏰 RAG con soberanía total: OCR + pgvector + Gemma, todo tuyo

**Curso práctico · ~100 minutos · Google Colab (GPU) + una VM de GCloud**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_selfhosted_pgvector.ipynb)

El cierre de la serie. Cada curso quitó una dependencia; **este las quita todas de golpe**. El pipeline entero corre en infraestructura que tú controlas:

| Paso | Antes (servicio de Google) | **Ahora (tuyo)** |
|---|---|---|
| Extraer el texto del PDF | Document AI | **GOT-OCR 2.0** (open-source) en la GPU |
| Trocear (chunking) | reglas / Document AI | **Gemma** troceando con IA (contextual) |
| Almacén de vectores | BigQuery | **PostgreSQL + `pgvector`** en una VM |
| Embeddings | Vertex | **`BAAI/bge-m3`** en la GPU |
| Generación | Gemini | **Gemma 4 (E4B)** en la GPU |

> ⚠️ **Necesitas GPU** (*Entorno de ejecución → GPU*; T4 vale) **y un proyecto GCP con facturación** (crearemos una VM pequeña y **la borraremos al final**).

> El único "tercero" es **GCloud como IaaS** (nos alquila la VM). Ningún servicio de IA gestionado: el documento nunca sale de tu perímetro, de principio a fin.


## 🎒 Kit de supervivencia (si es tu primera vez con RAG)

- **Embedding** — texto → **vector** que captura su significado; textos parecidos, vectores cercanos.
- **Vector store** — base de datos que guarda vectores y encuentra los más cercanos a una consulta. Antes BigQuery; **hoy Postgres + `pgvector`**.
- **Chunk** — trozo de documento que se vectoriza por separado. Aquí los hace **una IA** (Gemma), no reglas.
- **RAG** — **(1)** recuperar los chunks relevantes, **(2)** pegárselos al modelo como contexto, **(3)** generar la respuesta con citas.

> El detalle a fondo está en los cursos anteriores ([emails](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_emails_bigquery_v2.ipynb), [PDF Document AI](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_pdf_polizas_bigquery.ipynb), [PDF OCR open-source](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_pdf_ocr_opensource.ipynb)).


---
## ⚖️ ¿Por qué montarlo todo uno mismo?

Un servicio gestionado es cómodo, pero el dato **sale de tu perímetro**, dependes del proveedor y pagas por uso. En seguros, salud o banca eso choca de frente con el RGPD y con muchos contratos. Montarlo tú cuesta trabajo, pero te da **soberanía**: el dato no se mueve, los modelos son tuyos (pesos que puedes guardar) y el coste es el de una máquina que controlas.

> 🎓 **Tesis del curso:** self-hosted no es "mejor" — es **una decisión de arquitectura con trade-offs**, y saber montarlo te da la opción. Al final (bloque 8) lo comparamos con honestidad.

### Requisitos y agenda

- GPU en Colab + proyecto GCP con facturación. No hace falta BigQuery, Vertex ni Document AI.

| # | Bloque | ⏱️ |
|---|--------|----|
| 0 | Setup | 8 min |
| 1 | Fabricar las pólizas en PDF | 5 min |
| 2 | **Extraer el texto con GOT-OCR 2.0** | 15 min |
| 3 | Cargar Gemma 4 | 5 min |
| 4 | **Trocear con IA** (Gemma + contexto de títulos) | 15 min |
| 5 | **Postgres + pgvector en una VM** | 20 min |
| 6 | Embeddings open-source → pgvector | 10 min |
| 7 | Búsqueda vectorial en SQL | 6 min |
| 8 | RAG + comparativa self-hosted vs. gestionado | 12 min |
| 9 | Evaluación y **borrar la VM** | 8 min |


---
# 0 · Setup ⏱️ ~8 min

▶️ **Qué hace esta celda:** instala todo lo local. Nada de BigQuery ni Vertex. Fijamos **`transformers >= 5.5`** porque es lo que necesita **Gemma 4** — y, afortunadamente, la misma versión soporta **GOT-OCR 2.0** (por eso no usamos el OCR del curso 3, que fijaba una versión vieja e incompatible).

> ⚠️ Si al cargar un modelo más abajo ves un error de versión, ve a *Entorno de ejecución → Reiniciar entorno* y reejecuta desde aquí (al actualizar `transformers` a veces hay que reiniciar).

In [ ]:
%pip install -q "transformers>=5.5,<6" accelerate bitsandbytes sentence-transformers \
    psycopg2-binary reportlab pymupdf pillow
print("✅ Dependencias instaladas")

▶️ **Qué hace esta celda:** comprueba la GPU.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ No hay GPU. Entorno de ejecución → Cambiar tipo de entorno → GPU.")
gpu = torch.cuda.get_device_properties(0)
print(f"✅ GPU: {gpu.name} · {gpu.total_memory/1e9:.0f} GB")

▶️ **Qué hace esta celda:** te autentica en GCloud y habilita **Compute Engine** (la VM) e **IAP** (el túnel seguro para hablar con Postgres sin abrirlo a internet).

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("✅ Autenticado")

PROJECT_ID = "tu-proyecto-gcp"  # @param {type:"string"}
!gcloud config set project {PROJECT_ID} --quiet
!gcloud services enable compute.googleapis.com iap.googleapis.com --quiet
print("✅ APIs Compute + IAP habilitadas")

> 🚩 **CHECKPOINT 1** — GPU detectada, proyecto fijado, APIs habilitadas.

---
# 1 · Fabricamos las pólizas en PDF ⏱️ ~5 min

Idéntico a los cursos de PDF: tres condicionados con numeración, tablas y la sección **4. EXCLUSIONES** con texto parecido al de **3. COBERTURAS**.

▶️ **Qué hace esta celda:** estilos y plantilla del documento.

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (BaseDocTemplate, PageTemplate, Frame, Paragraph,
                                Spacer, Table, TableStyle, PageBreak)
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER

ASEGURADORA = "Peñalara Seguros, S.A."

ss = getSampleStyleSheet()
H1 = ParagraphStyle("H1x", parent=ss["Heading1"], fontSize=15, spaceAfter=10,
                    textColor=colors.HexColor("#1a3d5c"))
H2 = ParagraphStyle("H2x", parent=ss["Heading2"], fontSize=12, spaceBefore=10,
                    spaceAfter=6, textColor=colors.HexColor("#2c5f8a"))
H3 = ParagraphStyle("H3x", parent=ss["Heading3"], fontSize=10.5, spaceBefore=8,
                    spaceAfter=4, textColor=colors.HexColor("#444444"))
BODY = ParagraphStyle("BODYx", parent=ss["BodyText"], fontSize=9.5, leading=13,
                      alignment=TA_JUSTIFY, spaceAfter=5)
# 👇 la famosa "letra pequeña": legalmente válida, visualmente hostil
SMALL = ParagraphStyle("SMALLx", parent=BODY, fontSize=6.5, leading=8.5,
                       textColor=colors.HexColor("#555555"))
TITLE = ParagraphStyle("TITLEx", parent=ss["Title"], fontSize=22,
                       textColor=colors.HexColor("#1a3d5c"))
CENTER = ParagraphStyle("CENTERx", parent=BODY, alignment=TA_CENTER)

def _tabla(data, col_widths):
    t = Table(data, colWidths=col_widths, repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1a3d5c")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 8),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#eef3f8")]),
    ]))
    return t

class PolizaDoc(BaseDocTemplate):
    """Documento con encabezado y pie 'Página X de Y' en cada página."""
    def __init__(self, filename, producto, codigo, **kw):
        super().__init__(filename, pagesize=A4, **kw)
        self.producto, self.codigo = producto, codigo
        frame = Frame(2.2*cm, 2.2*cm, A4[0]-4.4*cm, A4[1]-4.4*cm, id="n")
        self.addPageTemplates([PageTemplate(id="all", frames=[frame], onPage=self._decorar)])

    def _decorar(self, canvas, doc):
        canvas.saveState()
        canvas.setFont("Helvetica", 7)
        canvas.setFillColor(colors.HexColor("#777777"))
        canvas.drawString(2.2*cm, A4[1]-1.5*cm, f"{ASEGURADORA} · {self.producto}")
        canvas.drawRightString(A4[0]-2.2*cm, A4[1]-1.5*cm, f"Condicionado {self.codigo}")
        canvas.line(2.2*cm, A4[1]-1.65*cm, A4[0]-2.2*cm, A4[1]-1.65*cm)
        canvas.line(2.2*cm, 1.9*cm, A4[0]-2.2*cm, 1.9*cm)
        canvas.drawCentredString(A4[0]/2.0, 1.4*cm, f"Página {doc.page}")
        canvas.restoreState()

print("✅ Estilos y plantilla definidos")

▶️ **Qué hace esta celda:** la función que arma cada póliza.

In [ ]:
def construir_poliza(path, producto, codigo, version, coberturas, exclusiones, franquicias):
    story = []
    # ── Portada ──
    story += [Spacer(1, 4*cm), Paragraph(ASEGURADORA, CENTER), Spacer(1, 1*cm),
              Paragraph(producto, TITLE), Spacer(1, 0.6*cm),
              Paragraph("Condiciones Generales", CENTER), Spacer(1, 0.3*cm),
              Paragraph(f"Código de condicionado: <b>{codigo}</b> · Versión {version}", CENTER),
              PageBreak()]

    # ── 1. Definiciones ──
    story += [Paragraph("1. DEFINICIONES", H1),
              Paragraph("A efectos del presente contrato, se entiende por:", BODY)]
    for num, term, txt in [
        ("1.1", "Asegurado", "Persona física o jurídica titular del interés asegurado y sobre la que recaen las consecuencias económicas del siniestro."),
        ("1.2", "Tomador", "Persona que suscribe el contrato con el Asegurador y a quien corresponden las obligaciones derivadas del mismo."),
        ("1.3", "Siniestro", "Todo hecho cuyas consecuencias estén total o parcialmente cubiertas por las garantías de esta póliza."),
        ("1.4", "Franquicia", "Cantidad que queda a cargo del Asegurado en cada siniestro y que se deduce de la indemnización."),
        ("1.5", "Suma asegurada", "Límite máximo de indemnización por siniestro y anualidad de seguro."),
    ]:
        story += [Paragraph(f"{num} {term}", H3), Paragraph(txt, BODY)]

    # ── 2. Objeto ──
    story += [PageBreak(), Paragraph("2. OBJETO DEL SEGURO", H1),
              Paragraph(f"El Asegurador garantiza, dentro de los límites del presente condicionado, "
                        f"las consecuencias económicas de los riesgos descritos en la sección 3, hasta "
                        f"las sumas fijadas en las Condiciones Particulares de la póliza {producto}.", BODY),
              Paragraph("2.1 Ámbito territorial", H3),
              Paragraph("Las garantías surten efecto en el territorio español, salvo indicación expresa "
                        "en contrario en las Condiciones Particulares.", BODY),
              Paragraph("2.2 Ámbito temporal", H3),
              Paragraph("Quedan cubiertos los siniestros ocurridos durante la vigencia de la póliza y "
                        "declarados conforme a los plazos de la sección 6.", BODY)]

    # ── 3. Coberturas (CON TABLA) ──
    story += [PageBreak(), Paragraph("3. COBERTURAS", H1),
              Paragraph("Quedan cubiertas las siguientes garantías, con los límites indicados:", BODY),
              Spacer(1, 0.3*cm),
              _tabla([["Garantía", "Límite por siniestro", "Franquicia"]] + coberturas,
                     [7.5*cm, 4.5*cm, 3.5*cm]), Spacer(1, 0.4*cm)]
    for i, (gar, lim, fr) in enumerate(coberturas, start=1):
        story += [Paragraph(f"3.{i} {gar}", H3),
                  Paragraph(f"Se garantiza el pago de la indemnización por los daños directos "
                            f"ocasionados por {gar.lower()}, hasta el límite de {lim} por siniestro, "
                            f"con una franquicia de {fr}. La cobertura opera siempre que el hecho "
                            f"causante sea súbito, accidental e imprevisto para el Asegurado.", BODY)]

    # ── 4. Exclusiones (🎯 el gotcha) ──
    story += [PageBreak(), Paragraph("4. EXCLUSIONES", H1),
              Paragraph("<b>Con carácter general, y salvo pacto expreso en contrario, quedan "
                        "EXCLUIDOS de toda cobertura:</b>", BODY)]
    for i, (titulo, items) in enumerate(exclusiones, start=1):
        story += [Paragraph(f"4.{i} {titulo}", H2)]
        for letra, txt in zip("abcdefghij", items):
            story += [Paragraph(f"4.{i}.{letra}) {txt}", BODY)]
    story += [Spacer(1, 0.3*cm),
              Paragraph("Las exclusiones recogidas en la presente sección han sido específicamente "
                        "aceptadas por el Tomador mediante su firma en las Condiciones Particulares, "
                        "conforme al artículo 3 de la Ley 50/1980, de Contrato de Seguro, que exige "
                        "que las cláusulas limitativas de los derechos del Asegurado se destaquen de "
                        "modo especial y sean expresamente aceptadas por escrito.", SMALL)]

    # ── 5. Franquicias (otra tabla) ──
    story += [PageBreak(), Paragraph("5. FRANQUICIAS", H1),
              Paragraph("Se aplicarán las siguientes franquicias por modalidad:", BODY),
              Spacer(1, 0.3*cm),
              _tabla([["Modalidad", "Franquicia general", "Franquicia específica"]] + franquicias,
                     [6*cm, 4.75*cm, 4.75*cm])]

    # ── 6. Siniestros ──
    story += [PageBreak(), Paragraph("6. DECLARACIÓN Y TRAMITACIÓN DE SINIESTROS", H1),
              Paragraph("6.1 Plazo de comunicación", H3),
              Paragraph("El Tomador deberá comunicar el siniestro al Asegurador en el plazo máximo de "
                        "<b>siete (7) días</b> desde que tuviera conocimiento del mismo.", BODY),
              Paragraph("6.2 Documentación exigible", H3),
              Paragraph("Deberá aportarse: declaración del siniestro, acreditación de la titularidad "
                        "del bien, presupuesto o factura de reparación y, cuando proceda, atestado.", BODY),
              Paragraph("6.3 Peritación", H3),
              Paragraph("En caso de desacuerdo sobre la valoración, cada parte designará un perito. De "
                        "persistir la discrepancia, se designará un tercer perito de común acuerdo.", BODY),
              Paragraph("6.4 Pago de la indemnización", H3),
              Paragraph("El Asegurador abonará la indemnización en el plazo de cuarenta (40) días desde "
                        "la recepción de la declaración del siniestro.", BODY)]

    # ── 7. Prima ──
    story += [PageBreak(), Paragraph("7. PRIMA, DURACIÓN Y RENOVACIÓN", H1),
              Paragraph("7.1 Pago de la prima", H3),
              Paragraph("La prima es anual y pagadera por anticipado.", BODY),
              Paragraph("7.2 Impago y suspensión", H3),
              Paragraph("En caso de impago de la segunda o sucesivas primas, la cobertura quedará "
                        "<b>suspendida un mes después</b> del día de su vencimiento.", BODY),
              Paragraph("7.3 Duración y prórroga", H3),
              Paragraph("El contrato se prorrogará tácitamente por periodos anuales, salvo oposición "
                        "notificada con <b>un (1) mes</b> de antelación por el Tomador o <b>dos (2) "
                        "meses</b> por el Asegurador.", BODY)]

    PolizaDoc(path, producto, codigo).multiBuild(story)
    return path

print("✅ Constructor de pólizas definido")

▶️ **Qué hace esta celda:** el catálogo de 3 productos y la generación.

In [ ]:
import os

POLIZAS = [
    dict(
        path="HOGAR_PLUS.pdf", producto="Hogar Plus", codigo="HP-2026-01", version="3.2",
        coberturas=[
            ["Incendio, rayo y explosión", "300.000 €", "Sin franquicia"],
            ["Daños por agua por rotura accidental de conducciones", "50.000 €", "150 €"],
            ["Robo y expoliación en el interior de la vivienda", "30.000 €", "150 €"],
            ["Rotura de cristales y vitrocerámica", "3.000 €", "Sin franquicia"],
            ["Responsabilidad civil familiar", "150.000 €", "300 €"],
            ["Fenómenos atmosféricos (viento, pedrisco, nieve)", "100.000 €", "300 €"],
        ],
        exclusiones=[
            ("Daños por agua no cubiertos", [
                "Los daños causados por <b>humedades, condensación o filtraciones</b> a través de muros, "
                "fachadas, terrazas o cubiertas, aun cuando sean consecuencia de lluvia, nieve o granizo.",
                "Los daños derivados de <b>falta de mantenimiento</b> de las conducciones, así como la "
                "corrosión, el óxido o el desgaste paulatino de tuberías.",
                "El coste de <b>localización y reparación de la avería</b> cuando no se haya producido "
                "daño material indemnizable.",
                "Los daños por <b>agua de lluvia que penetre por ventanas, puertas o huecos dejados "
                "abiertos</b> o defectuosamente cerrados por el Asegurado.",
            ]),
            ("Exclusiones generales", [
                "Los daños causados con dolo o culpa grave del Asegurado.",
                "Los daños derivados de <b>vicio propio o defecto de construcción</b> preexistente.",
                "Los daños calificados como catástrofe nacional o cubiertos por el <b>Consorcio de "
                "Compensación de Seguros</b>.",
                "Los daños en <b>viviendas deshabitadas</b> más de 60 días consecutivos.",
            ]),
        ],
        franquicias=[["Vivienda habitual", "150 €", "300 € en RC familiar"],
                     ["Segunda residencia", "300 €", "600 € en daños por agua"],
                     ["Vivienda en alquiler", "300 €", "600 € en robo"]],
    ),
    dict(
        path="AUTO_TODO_RIESGO.pdf", producto="Auto Todo Riesgo", codigo="AT-2026-04", version="2.1",
        coberturas=[
            ["Responsabilidad civil obligatoria", "Ilimitada (legal)", "Sin franquicia"],
            ["Daños propios por colisión o vuelco", "Valor venal + 20%", "300 €"],
            ["Robo total o parcial del vehículo", "Valor venal", "300 €"],
            ["Incendio del vehículo", "Valor venal", "Sin franquicia"],
            ["Lunas (parabrisas, laterales y trasera)", "Sin límite", "Sin franquicia"],
            ["Asistencia en viaje desde kilómetro 0", "Incluida", "Sin franquicia"],
        ],
        exclusiones=[
            ("Circunstancias del conductor", [
                "Siniestros conduciendo bajo <b>influencia de bebidas alcohólicas</b>, drogas o estupefacientes.",
                "Siniestros cuando el conductor <b>carezca de permiso de conducción</b> en vigor.",
                "Siniestros en <b>carreras, apuestas o pruebas deportivas</b> y sus entrenamientos.",
            ]),
            ("Uso del vehículo", [
                "El uso como <b>autoescuela, alquiler sin conductor, taxi o VTC</b>, salvo declaración expresa.",
                "El transporte de <b>mercancías peligrosas</b> o de más ocupantes de los autorizados.",
                "Los daños circulando por <b>vías no aptas</b> para la circulación o fuera de calzada.",
            ]),
            ("Daños no indemnizables", [
                "El <b>desgaste, uso o defecto de conservación</b> de las piezas.",
                "Los daños <b>exclusivamente estéticos</b> que no afecten a la seguridad.",
                "La <b>depreciación</b> del vehículo tras la reparación.",
            ]),
        ],
        franquicias=[["Conductor > 25 años y > 2 años de carné", "300 €", "Sin franquicia en lunas"],
                     ["Conductor novel (< 2 años de carné)", "600 €", "600 € en daños propios"],
                     ["Conductor ocasional no declarado", "900 €", "900 € en daños propios"]],
    ),
    dict(
        path="SALUD_FAMILIAR.pdf", producto="Salud Familiar", codigo="SF-2026-02", version="1.4",
        coberturas=[
            ["Medicina primaria y especialidades", "Sin límite", "Sin franquicia"],
            ["Pruebas diagnósticas (analítica, radiología)", "Sin límite", "Sin franquicia"],
            ["Hospitalización y cirugía en centros concertados", "Sin límite", "Sin franquicia"],
            ["Urgencias 24 h en cuadro médico", "Sin límite", "Sin franquicia"],
            ["Fisioterapia y rehabilitación", "30 sesiones/año", "10 € por sesión"],
            ["Psicología clínica", "20 sesiones/año", "15 € por sesión"],
        ],
        exclusiones=[
            ("Periodos de carencia", [
                "Las <b>intervenciones quirúrgicas</b> tienen una carencia de <b>seis (6) meses</b>.",
                "El <b>parto y la asistencia al embarazo</b> tienen una carencia de <b>diez (10) meses</b>.",
                "Los <b>tratamientos de reproducción asistida</b> tienen una carencia de <b>veinticuatro "
                "(24) meses</b> y se limitan a tres ciclos.",
            ]),
            ("Prestaciones no cubiertas", [
                "Las <b>enfermedades preexistentes</b> no declaradas en el cuestionario de salud.",
                "La <b>cirugía estética</b> y todo tratamiento sin finalidad terapéutica.",
                "Los tratamientos de <b>odontología</b> salvo extracción y limpieza anual.",
                "Los <b>medicamentos y prótesis</b> no incluidos en el catálogo.",
                "La asistencia <b>fuera del cuadro médico</b>, salvo urgencia vital acreditada.",
            ]),
        ],
        franquicias=[["Modalidad sin copago", "Sin franquicia", "Sin franquicia"],
                     ["Modalidad con copago", "Según acto médico", "10 € consulta / 25 € urgencia"],
                     ["Modalidad reembolso", "20% del gasto", "Límite 60.000 €/año"]],
    ),
]

os.makedirs("polizas", exist_ok=True)
for p in POLIZAS:
    construir_poliza(os.path.join("polizas", p["path"]), p["producto"], p["codigo"],
                     p["version"], p["coberturas"], p["exclusiones"], p["franquicias"])
print(f"✅ {len(POLIZAS)} pólizas generadas en ./polizas/")

---
# 2 · Extraer el texto con GOT-OCR 2.0 ⏱️ ~15 min

> 🧠 **Por qué OCR y no `pypdf`.** En el [curso 3](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_pdf_ocr_opensource.ipynb) demostramos que `pypdf` destroza tablas y falla con escaneados. Aquí usamos **GOT-OCR 2.0**, un OCR open-source (580M) **integrado en `transformers`** que *entiende* la página y devuelve **markdown** (encabezados, tablas). Que sea nativo de la librería es lo que le permite convivir con el `transformers` moderno que necesita Gemma 4 — cosa que el OCR del curso 3 no podía.

Rasterizamos cada página a imagen (como si fuera un escaneo) y la pasamos por el modelo con `format=True` para obtener markdown.

▶️ **Qué hace esta celda:** carga GOT-OCR 2.0 en la GPU (descarga ~1,4 GB la primera vez). En T4 usamos atención `sdpa` (la T4 no tiene FlashAttention).

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

OCR_ID = "stepfun-ai/GOT-OCR-2.0-hf"
print("⏳ Cargando GOT-OCR 2.0...")
ocr_model = AutoModelForImageTextToText.from_pretrained(
    OCR_ID, device_map="auto", attn_implementation="sdpa")
ocr_proc = AutoProcessor.from_pretrained(OCR_ID, use_fast=True)
print("✅ GOT-OCR 2.0 cargado")

▶️ **Qué hace esta celda:** rasteriza un PDF a imágenes (200 DPI, el punto dulce para GOT-OCR2, que reescala a 1024 px) y define la función de OCR de una página → markdown.

In [ ]:
import fitz  # PyMuPDF
from PIL import Image

def pdf_a_imagenes(pdf_path, dpi=200):
    doc = fitz.open(pdf_path)
    imgs = []
    for page in doc:
        pix = page.get_pixmap(dpi=dpi)
        imgs.append(Image.frombytes("RGB", (pix.width, pix.height), pix.samples))
    doc.close()
    return imgs

def ocr_pagina(img):
    """Una imagen de página → markdown (encabezados, tablas) con GOT-OCR2 en modo 'format'."""
    inputs = ocr_proc(img, return_tensors="pt", format=True).to(ocr_model.device)
    ids = ocr_model.generate(**inputs, do_sample=False, tokenizer=ocr_proc.tokenizer,
                             stop_strings="<|im_end|>", max_new_tokens=4096)
    return ocr_proc.decode(ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# vistazo: la página de coberturas de Hogar Plus, como imagen
from IPython.display import display
imgs_demo = pdf_a_imagenes("polizas/HOGAR_PLUS.pdf")
print(f"✅ {len(imgs_demo)} páginas rasterizadas")
display(imgs_demo[3].resize((360, int(360*imgs_demo[3].height/imgs_demo[3].width))))

▶️ **Qué hace esta celda:** pasa **todas las páginas** de las 3 pólizas por el OCR. ⏳ Es lento en T4 (varios min por póliza): momento ☕. Guardamos el markdown por página (para rastrear la página al citar).

In [ ]:
ocr_por_doc = {}
for p in POLIZAS:
    nombre = p["path"].replace(".pdf", "")
    print(f"⏳ OCR de {nombre}...")
    ocr_por_doc[nombre] = [ocr_pagina(im) for im in pdf_a_imagenes(f"polizas/{p['path']}")]
    print(f"   ✅ {len(ocr_por_doc[nombre])} páginas")

▶️ **Qué hace esta celda:** enseña **lo que leyó el OCR** en la página de coberturas — markdown, con su encabezado y la tabla. Y **libera GOT-OCR2 de la GPU**: ya no lo necesitamos y hay que dejar sitio para Gemma.

In [ ]:
print("═══ Markdown de la página de COBERTURAS (Hogar Plus) ═══\n")
print(ocr_por_doc["HOGAR_PLUS"][3][:1500])

# liberar la VRAM del OCR antes de cargar Gemma
import gc
del ocr_model, ocr_proc
gc.collect(); torch.cuda.empty_cache()
print("\n🧹 GOT-OCR2 liberado de la GPU")

> 🚩 **CHECKPOINT 2** — Deberías ver texto markdown legible (encabezados, tabla de garantías). Si sale ilegible, revisa la GPU. *(Nota honesta: GOT-OCR2 es muy bueno en texto y tablas simples; en tablas muy enrevesadas puede fallar. Para pólizas va sobrado, y además el troceado con IA del bloque 4 tolera imperfecciones.)*

---
# 3 · Cargar Gemma 4 ⏱️ ~5 min

Cargamos **Gemma 4 (E4B)** una sola vez: lo usaremos para **dos** cosas — trocear con IA (bloque 4) y generar las respuestas (bloque 8). En 4-bit para caber en la T4.

▶️ **Qué hace esta celda:** carga Gemma en 4-bit y define un ayudante `gemma_texto()` para pedirle cosas (lo reutilizan el troceador y el RAG).

> 💡 Alternativa clásica: `google/gemma-3-4b-it` (4B denso). Ojo, ese está *gated* (aceptar licencia + `login()` con token).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

GEMMA_ID = "google/gemma-4-E4B-it"   # Apache 2.0, sin token. Alternativa: "google/gemma-3-4b-it"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
print("⏳ Cargando Gemma en 4-bit...")
gemma_tok = AutoTokenizer.from_pretrained(GEMMA_ID)
gemma = AutoModelForCausalLM.from_pretrained(
    GEMMA_ID, quantization_config=bnb, device_map="auto", attn_implementation="eager")

def gemma_texto(prompt, max_new_tokens=512):
    """Envía un prompt a Gemma con su chat template y devuelve el texto."""
    inputs = gemma_tok.apply_chat_template(
        [{"role": "user", "content": prompt}], add_generation_prompt=True,
        tokenize=True, return_tensors="pt", return_dict=True).to(gemma.device)
    out = gemma.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return gemma_tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

print("✅ Gemma cargado")

---
# 4 · Trocear con IA ⏱️ ~15 min

En vez de cortar por reglas, dejamos que **Gemma** trocee cada página en **cláusulas autocontenidas**, y —lo importante— le **anteponemos el contexto de los títulos anteriores** (*contextual retrieval*): un fragmento de la página 5 debe saber que cuelga de "4. EXCLUSIONES" aunque esa palabra ya no esté en esa página.

**Dos piezas:**
1. Un rastreador de **jerarquía de títulos** (breadcrumb "4 EXCLUSIONES › 4.1 Daños por agua") que mantenemos entre páginas.
2. Gemma, que divide el texto de cada página en fragmentos y les antepone esa sección.

> 🎓 Es la misma idea que el chunking con IA que vimos en el curso de emails y en el opcional del de PDF — pero aquí con **Gemma local**, no Gemini. Nada sale de la máquina.

▶️ **Qué hace esta celda:** define el rastreador de breadcrumb y el troceador con IA. Por cada página: actualiza la jerarquía de títulos y pide a Gemma los fragmentos en JSON, con la sección antepuesta. Tiene *fallback* por si el modelo se desvía del JSON.

In [ ]:
import re, json

def _actualizar_breadcrumb(pila, linea):
    """Detecta un título (markdown # o numeración 4.1) y actualiza la pila de secciones."""
    m = re.match(r"^\s*#{0,6}\s*(\d+(?:\.\d+)*)\.?\s+([A-ZÁÉÍÓÚÑ][^\n]{2,70})\s*$", linea)
    if not m:
        return
    numero, titulo = m.group(1), m.group(2).strip()
    nivel = numero.count(".") + 1
    del pila[nivel-1:]
    while len(pila) < nivel-1:
        pila.append("…")
    pila.append(f"{numero} {titulo}")

def _extraer_json_array(resp):
    resp = re.sub(r"```(?:json)?", "", resp).strip()
    try:
        return json.loads(resp[resp.index("["):resp.rindex("]")+1])
    except Exception:
        return None

def trocear_con_ia(paginas_md, documento):
    filas, pila, n = [], [], 0
    for pagina, md_pag in enumerate(paginas_md, start=1):
        contexto = " › ".join(pila) if pila else "(inicio del documento)"
        # f-string SOLO para 'contexto' (nuestro, seguro); el markdown va concatenado
        # (puede contener llaves de LaTeX y romper un .format)
        prompt = (
            "Eres un experto troceando documentos para un buscador. Divide el TEXTO de esta "
            "página de un condicionado de seguro en fragmentos AUTOCONTENIDOS, uno por cada "
            "cláusula o idea. A CADA fragmento antepónle, entre corchetes, la sección a la que "
            "pertenece (usa los títulos del propio texto; si la página continúa una sección "
            f"anterior, esa sección es: {contexto}). Devuelve SOLO un array JSON de strings, "
            "sin nada más.\n\nTEXTO:\n" + md_pag)
        salida = gemma_texto(prompt, max_new_tokens=1200)
        frags = _extraer_json_array(salida) or [f"[{contexto}] {md_pag}"]
        # actualizar la jerarquía con los títulos de esta página (afectan a las siguientes)
        for linea in md_pag.split("\n"):
            _actualizar_breadcrumb(pila, linea)
        heading = " › ".join(pila) if pila else contexto
        for frag in frags:
            n += 1
            filas.append({"documento": documento, "chunk_id": f"{documento}-{n:04d}",
                          "pagina": pagina, "heading": heading, "content": str(frag)})
    return filas

print("✅ Troceador con IA definido")

▶️ **Qué hace esta celda:** trocea las 3 pólizas con Gemma. ⏳ Otra pausa ☕ (una llamada al modelo por página).

In [ ]:
import pandas as pd

filas = []
for nombre, paginas in ocr_por_doc.items():
    print(f"⏳ Troceando {nombre} con IA...")
    filas += trocear_con_ia(paginas, nombre)
df_chunks = pd.DataFrame(filas)
print(f"✅ {len(df_chunks)} chunks")
df_chunks.groupby("documento").size().to_frame("chunks")

▶️ **Qué hace esta celda:** el momento de la verdad. Los chunks sobre **agua** de Hogar Plus, con la sección que la IA les antepuso.

In [ ]:
pd.set_option("display.max_colwidth", 100)
agua = df_chunks[(df_chunks.documento == "HOGAR_PLUS") &
                 (df_chunks.content.str.contains("agua", case=False))]
agua[["pagina", "heading"]].assign(extracto=agua.content.str[:120])

> 🚩 **CHECKPOINT 3** — Deberías ver fragmentos de "agua" con secciones distintas (coberturas y exclusiones) y **autocontenidos**. Esa distinción —puesta por la IA— es la que salvará al RAG.

---
# 5 · Postgres + pgvector en una VM ⏱️ ~20 min

Nuestro **almacén de vectores**: una **VM de Compute Engine** con **PostgreSQL** y la extensión **`pgvector`** (que añade el tipo `vector` y la búsqueda por similitud a Postgres). Nos conectamos por un **túnel IAP** — sin exponer el 5432 a internet.

> 🧠 **Por qué IAP:** la IP del Colab cambia y abrir Postgres a internet es pedir problemas. IAP tuneliza el puerto por tu identidad de Google. Defensa en capas: IAM + firewall al rango IAP + contraseña.

▶️ **Qué hace esta celda:** escribe el **startup-script** que se ejecuta dentro de la VM al arrancar: añade el repo oficial de PostgreSQL (fija la v16, reproducible), instala `pgvector`, crea la base y el usuario, y habilita `CREATE EXTENSION vector`.

In [ ]:
%%writefile startup-postgres-pgvector.sh
#!/bin/bash
set -euxo pipefail
export DEBIAN_FRONTEND=noninteractive
PGVER=16

DB_PASS="$(curl -s -H 'Metadata-Flavor: Google' \
  'http://metadata.google.internal/computeMetadata/v1/instance/attributes/db-password')"

apt-get update
apt-get install -y curl ca-certificates postgresql-common
yes | /usr/share/postgresql-common/pgdg/apt.postgresql.org.sh
apt-get update
apt-get install -y "postgresql-${PGVER}" "postgresql-${PGVER}-pgvector"

sudo -u postgres psql -v ON_ERROR_STOP=1 <<SQL
CREATE ROLE appuser LOGIN PASSWORD '${DB_PASS}';
CREATE DATABASE appdb OWNER appuser;
SQL
sudo -u postgres psql -v ON_ERROR_STOP=1 -d appdb -c "CREATE EXTENSION IF NOT EXISTS vector;"

CONF="/etc/postgresql/${PGVER}/main"
echo "listen_addresses = '*'" >> "${CONF}/postgresql.conf"
echo "host all all 35.235.240.0/20 scram-sha-256" >> "${CONF}/pg_hba.conf"
systemctl restart postgresql

curl -s -X PUT --data "ready" -H 'Metadata-Flavor: Google' \
  "http://metadata.google.internal/computeMetadata/v1/instance/guest-attributes/startup/status"

▶️ **Qué hace esta celda:** crea la VM (`e2-medium`, Debian 12), abre el firewall **solo al rango IAP** en el 5432, y te concede el permiso de túnel. La contraseña es aleatoria y vive solo en esta sesión.

In [ ]:
import secrets, subprocess

DB_PASS = secrets.token_urlsafe(20)
ZONE, VM = "europe-west1-b", "pg-vector-vm"
EMAIL = subprocess.run(["gcloud", "config", "get-value", "account"],
                       capture_output=True, text=True).stdout.strip()

!gcloud compute instances create {VM} --zone={ZONE} --machine-type=e2-medium \
    --image-family=debian-12 --image-project=debian-cloud \
    --metadata=db-password={DB_PASS} \
    --metadata-from-file=startup-script=startup-postgres-pgvector.sh --quiet

!gcloud compute firewall-rules create allow-iap-postgres \
    --direction=INGRESS --action=allow --rules=tcp:5432 \
    --source-ranges=35.235.240.0/20 --quiet 2>/dev/null || echo "(la regla ya existía)"

!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member=user:{EMAIL} --role=roles/iap.tunnelResourceAccessor \
    --condition=None --quiet > /dev/null
print("✅ VM creada · firewall (solo IAP) · permiso de túnel concedido")

▶️ **Qué hace esta celda:** abre el **túnel IAP** en segundo plano y **espera a que Postgres esté listo de verdad** (el startup-script tarda 2-4 min; reintentamos hasta que responde). ☕

In [ ]:
import socket, time, os, signal, psycopg2

LOCAL_PORT = 5432
tunnel = subprocess.Popen(
    ["gcloud", "compute", "start-iap-tunnel", VM, "5432",
     f"--local-host-port=localhost:{LOCAL_PORT}", f"--zone={ZONE}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, preexec_fn=os.setsid)

def port_open(host, port, timeout=120):
    t0 = time.time()
    while time.time() - t0 < timeout:
        with socket.socket() as s:
            s.settimeout(2)
            if s.connect_ex((host, port)) == 0:
                return True
        time.sleep(2)
    return False
assert port_open("127.0.0.1", LOCAL_PORT), "El túnel IAP no abrió el puerto local"

def wait_db(timeout=420):
    t0, last = time.time(), None
    while time.time() - t0 < timeout:
        try:
            return psycopg2.connect(host="127.0.0.1", port=LOCAL_PORT, dbname="appdb",
                                    user="appuser", password=DB_PASS, connect_timeout=3)
        except Exception as e:
            last = e; time.sleep(6)
    raise TimeoutError(f"Postgres no respondió (¿sigue instalando?): {last}")

print("⏳ Esperando a que la VM instale Postgres + pgvector (2-4 min)...")
conn = wait_db(); conn.autocommit = True
cur = conn.cursor()
cur.execute("SELECT version()"); print("✅ Conectado:", cur.fetchone()[0][:45], "...")
cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
print("✅ pgvector disponible")

> 🚩 **CHECKPOINT 4** — `✅ Conectado` + `✅ pgvector disponible`. Si `wait_db` agota el tiempo, reejecuta esta celda (el túnel sigue vivo).

---
# 6 · Embeddings open-source → pgvector ⏱️ ~10 min

En vez de Vertex, **`BAAI/bge-m3`** (MIT) en la GPU: multilingüe, **1024 dimensiones**, contexto largo, sin prefijos.

▶️ **Qué hace esta celda:** carga `bge-m3`, crea la tabla con columna **`vector(1024)`**, vectoriza los chunks, los inserta y crea un índice **HNSW**.

In [ ]:
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer("BAAI/bge-m3", device="cuda")
print("⏳ Vectorizando", len(df_chunks), "chunks...")
embs = emb_model.encode(df_chunks["content"].tolist(),
                        normalize_embeddings=True, batch_size=16, show_progress_bar=True)

cur.execute("DROP TABLE IF EXISTS polizas_chunks")
cur.execute("""
    CREATE TABLE polizas_chunks (
        chunk_id  text PRIMARY KEY, documento text, pagina int,
        heading   text, content text, embedding vector(1024))
""")
for (_, row), emb in zip(df_chunks.iterrows(), embs):
    lit = "[" + ",".join(f"{x:.6f}" for x in emb) + "]"
    cur.execute("INSERT INTO polizas_chunks VALUES (%s,%s,%s,%s,%s,%s)",
                (row.chunk_id, row.documento, int(row.pagina), row.heading, row.content, lit))
cur.execute("CREATE INDEX ON polizas_chunks USING hnsw (embedding vector_cosine_ops)")
cur.execute("SELECT COUNT(*) FROM polizas_chunks")
print(f"✅ {cur.fetchone()[0]} chunks vectorizados y guardados en Postgres/pgvector")

> 🚩 **CHECKPOINT 5** — El número coincide con el del bloque 4. Ya tienes tu vector store **propio**.

---
# 7 · Búsqueda vectorial en SQL ⏱️ ~6 min

El `VECTOR_SEARCH` de BigQuery se convierte en **SQL de Postgres**. El operador **`<=>`** es la distancia coseno.

▶️ **Qué hace esta celda:** vectoriza la pregunta con `bge-m3` y pide a Postgres los k chunks más cercanos, con filtro opcional por póliza.

In [ ]:
def buscar_clausulas(pregunta, k=5, documento=None):
    q = emb_model.encode([pregunta], normalize_embeddings=True)[0]
    q_lit = "[" + ",".join(f"{x:.6f}" for x in q) + "]"
    where = "WHERE documento = %(doc)s" if documento else ""
    sql = f"""
        SELECT documento, pagina, heading, content,
               ROUND((embedding <=> %(q)s)::numeric, 4) AS distancia
        FROM polizas_chunks
        {where}
        ORDER BY embedding <=> %(q)s
        LIMIT %(k)s
    """
    params = {"q": q_lit, "k": k}
    if documento:
        params["doc"] = documento
    cur.execute(sql, params)
    return pd.DataFrame(cur.fetchall(),
                        columns=["documento", "pagina", "heading", "content", "distancia"])

buscar_clausulas("¿me cubre el agua de lluvia que ha entrado en casa?",
                 k=4, documento="HOGAR_PLUS")[["documento", "pagina", "heading", "distancia"]]

> 🎓 El buscador recupera chunks de **coberturas y de exclusiones** (semánticamente ambos hablan de agua); la distinción la pone el **heading** que la IA antepuso. Y todo ocurre ya **en tu Postgres**.

---
# 8 · RAG con Gemma + comparativa ⏱️ ~12 min

El último tercero fuera: en vez de Gemini, **Gemma** (ya cargado en el bloque 3) genera la respuesta a partir de lo recuperado de tu Postgres, citando.

▶️ **Qué hace esta celda:** el RAG completo, **100% local**. Recupera de Postgres, monta el contexto con la procedencia y Gemma responde citando.

In [ ]:
INSTR = ("Eres el asistente de atención al cliente de Peñalara Seguros. "
         "Responde SOLO con la información del CONTEXTO. "
         "Antes de afirmar que algo está cubierto, comprueba la SECCIÓN del fragmento: "
         "si viene de EXCLUSIONES, NO está cubierto, aunque hable del mismo riesgo. "
         "Cita siempre la póliza, la página y la cláusula. "
         "Si el contexto no basta, di que no consta en la póliza. No inventes.")

def responder(pregunta, k=6, documento=None, verbose=True):
    docs = buscar_clausulas(pregunta, k, documento)
    contexto = "\n\n---\n\n".join(
        f"[Póliza: {r.documento} | Página: {r.pagina} | Sección: {r.heading}]\n{r.content}"
        for r in docs.itertuples())
    if verbose:
        print(f"🔎 {len(docs)} fragmentos | secciones: {[s for s in docs.heading.unique() if s][:4]}\n")
    return gemma_texto(f"{INSTR}\n\n### CONTEXTO:\n{contexto}\n\n### PREGUNTA:\n{pregunta}", 512)

print(responder("Ha entrado agua de lluvia por la terraza y se me ha estropeado el parqué. "
                "¿Me lo cubre el seguro de hogar?", documento="HOGAR_PLUS"))

> 🎓 Si todo fue bien, Gemma responde **NO cubierto**, citando la exclusión — y **nada salió de tu infraestructura**: lo leyó GOT-OCR2, lo troceó Gemma, lo vectorizó bge-m3, lo buscó tu Postgres y lo respondió tu Gemma.

▶️ **Qué hacen estas dos celdas:** más preguntas, sobre otras pólizas.

In [ ]:
print(responder("Contraté el seguro de salud hace 3 meses y necesito operarme del menisco. "
                "¿Me lo cubren ya?", documento="SALUD_FAMILIAR"))

In [ ]:
print(responder("Tuve un accidente y el coche lo conducía mi sobrino, que sacó el carné hace 8 meses "
                "y no está declarado en la póliza. ¿Qué franquicia me toca pagar?",
                documento="AUTO_TODO_RIESGO"))

### Comparativa: self-hosted vs. gestionado

| Eje | Gestionado (BigQuery+Vertex+Gemini) | Self-hosted (este curso) |
|---|---|---|
| **Privacidad del dato** | sale a las APIs de Google | **no sale de tu perímetro** |
| **Coste** | por uso (tokens, páginas, consultas) | por **infraestructura** encendida (fijo) |
| **Escalado** | automático | lo gestionas tú |
| **Puesta en marcha** | minutos, cero infra | horas: VMs, drivers, versiones, VRAM |
| **Mantenimiento** | de Google | **tuyo** |
| **Calidad del modelo** | Gemini (frontera) | Gemma 4 (muy bueno, un escalón por debajo) |
| **Dependencia** | del proveedor | de nadie: guardas los pesos |

> 🎓 **Cuándo cada uno.** Gestionado: prototipos, dato no sensible, volumen irregular, sin equipo de infra. Self-hosted: **el dato no puede salir** (seguros, salud, banca, sector público), volumen alto y sostenido, o necesitas **soberanía**. No hay ganador universal; lo valioso es **saber montar los dos** y elegir por la restricción que más apriete.


---
# 9 · Evaluación y limpieza ⏱️ ~8 min

### Mini-evaluación: ¿acierta el sentido?

▶️ **Qué hace esta celda:** casos con respuesta conocida, donde solo la jerarquía desempata cobertura de exclusión — resueltos por tu stack self-hosted.

In [ ]:
casos = [
    ("Se ha roto una tubería del baño y ha inundado el salón. ¿Está cubierto?",
     "HOGAR_PLUS", "SÍ (cláusula 3.2, rotura accidental de conducciones)"),
    ("Entra agua de lluvia por una filtración en la terraza. ¿Está cubierto?",
     "HOGAR_PLUS", "NO (cláusula 4.1.a, filtraciones aunque sean por lluvia)"),
    ("Dejé la ventana abierta, llovió y se estropeó el suelo. ¿Está cubierto?",
     "HOGAR_PLUS", "NO (cláusula 4.1.d, agua por huecos dejados abiertos)"),
    ("Necesito una operación de rodilla a los 8 meses de contratar. ¿Cubierta?",
     "SALUD_FAMILIAR", "SÍ (carencia quirúrgica de 6 meses ya superada)"),
    ("Quiero una rinoplastia estética. ¿La cubre el seguro de salud?",
     "SALUD_FAMILIAR", "NO (exclusión: cirugía estética sin fin terapéutico)"),
]

for pregunta, doc, esperado in casos:
    print("═" * 78)
    print(f"❓ {pregunta}")
    print(f"🎯 Esperado: {esperado}")
    print(f"🤖 {responder(pregunta, documento=doc, verbose=False)[:300]}...\n")

### 🧪 Autoevaluación

1. ¿Por qué GOT-OCR 2.0 puede convivir con Gemma 4 en el mismo runtime y Unlimited-OCR no?
2. En el troceado con IA, ¿cómo consigue un fragmento de la página 5 "saber" que es una exclusión?
3. ¿Qué reemplaza a BigQuery, y con qué tipo de columna guardamos los vectores?
4. ¿Por qué conectamos a Postgres por túnel IAP en vez de abrir el 5432 a internet?
5. Enumera qué corre **en tu GPU** y qué corre **en la VM**.

### Ejercicios para casa

- **Fácil** — Sube `dpi` a 300 en el OCR y mira si mejora el markdown (y cuánto más tarda).
- **Medio** — Cambia `bge-m3` por `Qwen/Qwen3-Embedding-0.6B` (sus *queries* usan `prompt_name="query"`).
- **Medio** — En el troceador, guarda la jerarquía completa (sección › subsección › cláusula) en columnas separadas y filtra por sección.
- **Difícil** — Compara el troceado con IA contra el troceado por reglas del curso 3: ¿mejora el retrieval? Mídelo.
- **Difícil** — Sustituye Gemma por tu **adapter fine-tuneado** del cuaderno siguiente y compara.

### 🧹 Limpieza — ⚠️ IMPORTANTE

> Una VM encendida **cuesta mientras exista**. **Ejecuta esta celda al terminar** con `BORRAR = True`.

In [ ]:
BORRAR = False   # ⚠️ cámbialo a True y ejecuta para borrar la VM (si no, seguirá costando)

import gc
try:
    os.killpg(os.getpgid(tunnel.pid), signal.SIGTERM); print("🔌 Túnel IAP cerrado")
except Exception:
    pass

if BORRAR:
    !gcloud compute instances delete {VM} --zone={ZONE} --quiet
    !gcloud compute firewall-rules delete allow-iap-postgres --quiet
    print("🗑️  VM y regla de firewall borradas")
else:
    print("⚠️  BORRAR = False. La VM SIGUE ENCENDIDA y generando coste.")
    print(f"    Para apagarla: gcloud compute instances delete {VM} --zone={ZONE}")

try:
    del gemma, emb_model; gc.collect(); torch.cuda.empty_cache(); print("🧹 GPU liberada")
except Exception:
    pass

---
## 🧭 Mapa final: qué te llevas

1. **La soberanía se monta pieza a pieza** — OCR, chunking, vector store, embeddings y generación son sustituibles uno a uno.
2. **La compatibilidad de versiones es parte del diseño**: elegir modelos **nativos de la librería** (GOT-OCR2) en vez de código remoto con versión fijada es lo que permite que convivan.
3. **El chunking con IA + contexto de títulos** hace los fragmentos autocontenidos — la clave para que el RAG no confunda cobertura con exclusión.
4. **`pgvector` convierte Postgres en un vector store** capaz: el `VECTOR_SEARCH` es un `ORDER BY embedding <=> consulta`.
5. **El pipeline de RAG no cambió** — recuperar, aumentar, generar con citas. Solo cambió **dónde vive cada pieza**.

### 📚 Para seguir
- [GOT-OCR2](https://huggingface.co/stepfun-ai/GOT-OCR-2.0-hf) · [pgvector](https://github.com/pgvector/pgvector) · [bge-m3](https://huggingface.co/BAAI/bge-m3) · [Gemma 4](https://huggingface.co/google/gemma-4-E4B-it)
- **Siguiente:** fine-tuning de Gemma 4 para el estilo de Peñalara, enchufado a este mismo RAG.
